In [1]:
import pandas as pd
import sklearn
import seaborn as sns
import matplotlib.pyplot as plt

import pickle

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.metrics import root_mean_squared_error

In [9]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

import xgboost as xgb

In [2]:
import mlflow

mlflow.set_tracking_uri("sqlite:////workspaces/mlops/mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

with mlflow.start_run():
    mlflow.log_param("test_param", 1)
    mlflow.log_metric("test_metric", 0.5)

2026/06/07 15:51:13 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/07 15:51:13 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


In [3]:
trainlink = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet'
vallink = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-02.parquet'

In [4]:
def read_dataframe(filelink, n_data=1000):
    df = pd.read_parquet(filelink)
    df = df.sample(n=n_data, random_state=42)
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df["duration"] = df["duration"].dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    categorical = ['PULocationID', 'DOLocationID']
    #numerical = ['trip_distance']
    df[categorical] = df[categorical].astype(str)

    return df

In [5]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)

        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, "validation")],
            early_stopping_rounds=50
        )

        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)

        mlflow.log_metric("rmse", rmse)

        return {
            "loss": rmse,
            "status": STATUS_OK
        }

In [11]:
df_train = read_dataframe(trainlink)
df_val = read_dataframe(vallink, 100)

In [12]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [13]:
categorical = ['PU_DO'] # ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [14]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [15]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [16]:
search_space = {
    "max_depth": scope.int(hp.quniform("max_depth", 4, 100, 1)),
    "learning_rate": hp.loguniform("learning_rate", -3, 0),
    "reg_alpha": hp.loguniform("reg_alpha", -5, -1),
    "reg_lambda": hp.loguniform("reg_lambda", -6, -1),
    "min_child_weight": hp.loguniform("min_child_weight", -1, 3),
    "objective": "reg:squarederror",
    "seed": 42,
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials()
)

[0]	validation-rmse:9.03435                                                                                                                                                                                               
[1]	validation-rmse:8.39823                                                                                                                                                                                               
[2]	validation-rmse:7.87179                                                                                                                                                                                               
[3]	validation-rmse:7.45175                                                                                                                                                                                               
[4]	validation-rmse:7.10660                                                                                                 

In [24]:
mlflow.xgboost.autolog(disable=True)
with mlflow.start_run():
    best_params = {
        "learning_rate": 0.32312594192985483,
        "max_depth": 4,
        "min_child_weight": 0.8370351925367615,
        "objective": "reg:squarederror",
        "reg_alpha": 0.020041563446438926,
        "reg_lambda": 0.032662007646994694,
        "seed": 42,
    }
    
    mlflow.log_params(best_params)
    
    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, "validation")],
        early_stopping_rounds=50
    )
    
    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)
    with open("models/preprocessor", "wb") as f_out:
        pickle.dump(dv, f_out)
        
    mlflow.log_artifact("models/preprocessor", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")


[0]	validation-rmse:7.85606
[1]	validation-rmse:6.75535
[2]	validation-rmse:6.19389
[3]	validation-rmse:5.89090
[4]	validation-rmse:5.71849
[5]	validation-rmse:5.68455
[6]	validation-rmse:5.63254
[7]	validation-rmse:5.60461
[8]	validation-rmse:5.61748
[9]	validation-rmse:5.61845
[10]	validation-rmse:5.69138
[11]	validation-rmse:5.69240
[12]	validation-rmse:5.69278
[13]	validation-rmse:5.69363
[14]	validation-rmse:5.69174
[15]	validation-rmse:5.69250
[16]	validation-rmse:5.69287
[17]	validation-rmse:5.69365
[18]	validation-rmse:5.69452
[19]	validation-rmse:5.69354
[20]	validation-rmse:5.62557
[21]	validation-rmse:5.62594
[22]	validation-rmse:5.62584
[23]	validation-rmse:5.62666
[24]	validation-rmse:5.62701
[25]	validation-rmse:5.62627
[26]	validation-rmse:5.62622
[27]	validation-rmse:5.62657
[28]	validation-rmse:5.70225
[29]	validation-rmse:5.70289
[30]	validation-rmse:5.70306
[31]	validation-rmse:5.70377
[32]	validation-rmse:5.70560
[33]	validation-rmse:5.70628
[34]	validation-rmse:5.7

2026/06/07 16:34:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/opt/conda/envs/exp-tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [16:34:54] WARNING: /workspace/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
2026/06/07 16:34:57 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [27]:
xgboost_model = mlflow.xgboost.load_model("runs:/a9b9c78b40ac4a9ea6e91ad8de4e7b4a/models_mlflow")

In [31]:
y_pred = xgboost_model.predict(valid)
y_pred[:10]

array([ 5.9675636, 13.739661 ,  5.5271873, 21.67011  ,  7.5321784,
       28.105282 , 12.190053 ,  9.977894 , 20.419138 , 21.67011  ],
      dtype=float32)

In [33]:
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

2026/06/07 17:09:22 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/07 17:09:22 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


In [35]:
client.search_experiments()

[<Experiment: artifact_location='/workspaces/mlops/01-intro/mlruns/1', creation_time=1780835756340, experiment_id='1', last_update_time=1780835756340, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>,
 <Experiment: artifact_location='/workspaces/mlops/01-intro/mlruns/0', creation_time=1780835756329, experiment_id='0', last_update_time=1780835756329, lifecycle_stage='active', name='Default', tags={}>]

In [37]:
client.create_experiment(name="my-cool-experiment")

'2'

In [45]:
from mlflow.entities import ViewType
runs = client.search_runs(
    experiment_ids ="3",
    filter_string="metrics.rmse < 6.8",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"]
)

In [46]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

In [48]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [51]:
run_id = "ced1e4424321481185a6351b4045d16d"
model_uri = f"runs:/{run_id}/models_mlflow"
mlflow.register_model(model_uri=model_uri, name="nyc-taxi-regressor")

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...


MlflowException: Run with id=ced1e4424321481185a6351b4045d16d not found

In [52]:
client.list_registered_models()

AttributeError: 'MlflowClient' object has no attribute 'list_registered_models'

In [53]:
modle_name = "nyc-taxi-regressor"
latest_version = client.get_latest_varsions(name=modle_name)

for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

AttributeError: 'MlflowClient' object has no attribute 'get_latest_varsions'

In [ ]:
model_version = 4
new_stage = "Staging"
client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)

In [ ]:
from datetime import datetime
date = datetime.today()
client.update_model_version(
    name=model_name,
    version=model_version,
    description = f"The modle version {model_version} was transitioned to {new_stage} on {date}"
)

In [54]:
import pandas as pd
import mlflow

from sklearn.metrics import root_mean_squared_error

def read_dataframe(filename):
    df = pd.read_parquet(filename)
    df = df.sample(n=n_data, random_state=42)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df["duration"] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df["duration"] = df["duration"].dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ["PULocationID", "DOLocationID"]
    df[categorical] = df[categorical].astype(str)

    return df

def preprocess(df, dv):
    df["PU_DO"] = df["PULocationID"] + "_" + df["DOLocationID"]

    categorical = ["PU_DO"]
    numerical = ["trip_distance"]

    dicts = df[categorical + numerical].to_dict(orient="records")

    return dv.transform(dicts)

def test_model(name, stage, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")

    y_pred = model.predict(X_test)

    return {
        "rmse": root_mean_squared_error(y_test, y_pred)
    }

In [ ]:
df = pd.read_parquet(trainlink)

In [ ]:
client.download_artifacts(
    run_id=run_id,
    path="preprocessor",
    dst_path="."
)

In [ ]:
import pickle

with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

In [ ]:
X_test = preprocess(df, dv)

In [ ]:
target = "duration"
y_test = df[target].values

In [ ]:
%time test_model(name=model_name, stage="production", X_test=X_test, y_test=y_test)

In [ ]:
%time test_model(name=model_name, stage="Staging", X_test=X_test, y_test=y_test)

In [ ]:
client.transition_model_version_stage(
    name=model_name,
    version=4,
    stage="Production",
    archive_existing_versions=True
)